In [1]:
# Creating a mask for country specific BMR
# Run for each health variable

In [2]:
import os
import xarray as xr
import numpy as np

In [3]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"

In [4]:
# BMR from GBD (https://vizhub.healthdata.org/gbd-results/) is provided as a
# RATE per 100,000. i.e if the rate was 10 per 100,000 the value provided would
# be 10, not 0.0001
# To calculate the mortality we must divide the rate by 100,000 to convert to
# a per-person basis.

In [5]:
# === Health variables ===
# COPD (chronic obstructive pulonary disease)
# LRI (lower respiratory infection)
# IHD (ischemic heart disease)
# DM2 (type 2 diabetes)
# LC (tracheal, bronchus, and lung cancer)
# Stroke
health_VAR = "LC"

# Divide rate by 100,000 since rate is num per 100k
bmr_file = f"GBD_BMR_Country_{health_VAR}_newlabels_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
bmr = xr.open_dataarray(bmr_path) / 100000

# Use country masks to create a BMR mask for each country
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

In [6]:
# Create DataArray filled with NaNs
global_bmr_array = xr.DataArray(
    np.full((len(masks.lat), len(masks.lon)), np.nan),
    coords={"lat": masks.lat.values, "lon": masks.lon.values},
    dims=["lat", "lon"]
)

In [7]:
# Loop over countries, select the BMR for each country and apply to empty array
# using the country mask
for i in range(len(masks.country)):
    mask = masks.isel(country=i)
    country = masks.isel(country=i)["country"]
    bmr_country = bmr.sel(country=country)  # adds all three quantiles
    global_bmr_array = global_bmr_array.where(mask == 0, bmr_country)

In [8]:
# Save BMR mask
description = (f"Mean Baseline Mortality Rate per country for {health_VAR} "
               "from 1990-2009 as a global mask")
cite = ("Global Burden of Disease Collaborative Network. Global Burden of "
        "Disease Study 2021 (GBD 2021) Results. Seattle, United States: "
        "Institute for Health Metrics and Evaluation (IHME), 2022. Available "
        "from https://vizhub.healthdata.org/gbd-results/.")

global_bmr_array.attrs["description"] = description
global_bmr_array.attrs["citation"] = cite

out_file = f"GBD_BMR_Country_Mask_{health_VAR}_1990-2009.nc"
out_path = os.path.join(BMR_DIR, out_file)
global_bmr_array.to_netcdf(out_path)